# Image Data Augmentation Research Lab

Google Colab companion for the GitHub project.

Reference: Shorten & Khoshgoftaar (2019), *A survey on Image Data Augmentation for Deep Learning*.

In [ ]:
!pip -q install pillow numpy pandas matplotlib


In [ ]:
from google.colab import files
from PIL import Image, ImageEnhance
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()
filename = next(iter(uploaded))
original = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
print('Loaded:', filename, original.size)
display(original)

## Geometric augmentation

In [ ]:
flip = original.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
rotation = original.rotate(25, expand=True, fillcolor='white')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(axes, [original, flip, rotation], ['Original','Horizontal flip','Rotation']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.show()

## Colour-space augmentation

In [ ]:
brightness = ImageEnhance.Brightness(original).enhance(1.3)
contrast = ImageEnhance.Contrast(original).enhance(1.3)
saturation = ImageEnhance.Color(original).enhance(0.7)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, image, title in zip(axes, [original, brightness, contrast, saturation], ['Original','Brightness','Contrast','Saturation']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.show()

## Random erasing

In [ ]:
def random_erasing(image, area_percent=10, seed=2026):
    rng = np.random.default_rng(seed)
    array = np.asarray(image).copy()
    height, width, _ = array.shape
    target = max(1, int(height * width * area_percent / 100))
    ratio = float(rng.uniform(0.5, 2.0))
    erase_h = max(1, min(height, int(np.sqrt(target * ratio))))
    erase_w = max(1, min(width, int(np.sqrt(target / ratio))))
    y = int(rng.integers(0, max(1, height - erase_h + 1)))
    x = int(rng.integers(0, max(1, width - erase_w + 1)))
    array[y:y+erase_h, x:x+erase_w] = 0
    return Image.fromarray(array)

erased = random_erasing(original)
display(erased)

## Pixel-level diagnostics

In [ ]:
def mse(a, b):
    x = np.asarray(a.resize(b.size), dtype=np.float32)
    y = np.asarray(b, dtype=np.float32)
    return float(np.mean((x - y) ** 2))

def mae(a, b):
    x = np.asarray(a.resize(b.size), dtype=np.float32)
    y = np.asarray(b, dtype=np.float32)
    return float(np.mean(np.abs(x - y)))

def psnr(a, b):
    value = mse(a, b)
    return float('inf') if value == 0 else float(10 * np.log10((255.0 ** 2) / value))

pd.DataFrame([{
    'augmentation': 'random erasing',
    'MSE': mse(original, erased),
    'MAE': mae(original, erased),
    'PSNR_dB': psnr(original, erased)
}])

## Research boundary

Pixel-level diagnostics measure image change. They do not demonstrate improved CNN generalisation. A performance study should use a fixed model, controlled data split, repeated random seeds and task-specific evaluation metrics.